# Astra Galaxy 7 — Kaggle GPU training

Este notebook usa o ambiente GPU do Kaggle para executar o treinamento real do Astra. O modelo aprende dos dados; não há respostas pré-programadas.

In [ ]:
!nvidia-smi
!git clone -q https://github.com/andrelaerth44-pixel/Astra-Galaxy-7.git /kaggle/working/Astra-Galaxy-7
%cd /kaggle/working/Astra-Galaxy-7
!pip install -q -r requirements.txt


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
print('GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
assert torch.cuda.is_available(), 'GPU não está disponível nesta sessão do Kaggle.'


## Smoke test

Primeiro validamos o pipeline inteiro com um corpus pequeno. Depois substituímos o corpus de teste por um corpus real e aumentamos a configuração.

In [ ]:
!python scripts/make_sample_corpus.py --output data/raw/sample.jsonl
!python scripts/deduplicate_jsonl.py --input data/raw/sample.jsonl --output data/clean/sample.jsonl
!python scripts/split_corpus.py --input data/clean/sample.jsonl --train-output data/splits/train.jsonl --validation-output data/splits/validation.jsonl
!python scripts/prepare_data.py --input data/splits/train.jsonl --output data/processed/train.jsonl
!python scripts/prepare_data.py --input data/splits/validation.jsonl --output data/processed/validation.jsonl
!python scripts/build_tokenizer.py --input data/processed/train.jsonl --output data/tokenizer.json --vocab-size 256
!python scripts/tokenize_data.py --tokenizer data/tokenizer.json --input data/processed/train.jsonl --output data/processed/train.bin
!python scripts/tokenize_data.py --tokenizer data/tokenizer.json --input data/processed/validation.jsonl --output data/processed/validation.bin


In [ ]:
!python -m src.train --config config/smoke.yaml
!python -m src.evaluate --config config/smoke.yaml --checkpoint checkpoints/smoke/final.pt


## Próxima etapa

Depois do smoke test, use um corpus real legalmente utilizável e documentado. Não coloque segredos ou tokens no notebook. Checkpoints devem ser copiados para o armazenamento persistente do Kaggle ou publicados no GitHub apenas quando apropriado.